# Практика 3: Полная реализация трансформера

Соберём полный трансформер, включая энкодер и декодер. Начнём с реализации блока энкодера: создаём `nn.ModuleList` из нескольких слоёв внимания и `полносвязных слоёв FeedForward`, включающих два линейных преобразования и `активацию ReLU`. Добавляем `позиционные эмбеддинги` (т.к. трансформеры не обрабатывают последовательность напрямую, необходимо добавить информацию о позиции слов). Объединяем слои в полноценную сеть, `нормализуем с LayerNorm`. Далее переходим к декодеру: он похож на энкодер, но дополнительно использует механизм маскированного внимания для предсказания следующего токена.  `Энкодер - это BERT, Декодер - это GPT`. Применяем `оптимизатор AdamW`, используем `кросс-энтропийную функцию потерь`. Тестируем модель, проверяем логики её работы: подаём входной текст и анализируем `выходное распределение вероятностей по токенам`. Экспериментируем с различными настройками гиперпараметров (количество слоёв, размер скрытых представлений, число голов внимания) и анализируем влияние на качество.

In [1]:
import numpy as np
import os
import re
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from tqdm.auto import tqdm

from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

device


c:\Users\semen\works\NN2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

In [2]:
torch.cuda.empty_cache()

In [3]:
def make_causal_mask(T: int, device: torch.device) -> torch.Tensor:
    m = torch.tril(torch.ones(T, T, device=device))
    return m.unsqueeze(0).unsqueeze(0)

In [4]:
def make_padding_mask(tokens: torch.Tensor, pad_id: int) -> torch.Tensor:
    return (tokens != pad_id).unsqueeze(1).unsqueeze(2).float()

In [5]:
def scaled_dot_product_attention(Q, K, V, attn_mask=None, dropout_p=0.0, training=True):
    _, _, _, D = Q.shape
    scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(D)

    if attn_mask is not None:
        scores = scores.masked_fill(attn_mask == 0, float("-inf"))

    attn = torch.softmax(scores, dim=-1)

    if dropout_p > 0:
        attn = F.dropout(attn, p=dropout_p, training=training)

    out = torch.matmul(attn, V)
    return out, attn

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout_p: float = 0.0, bias: bool = True):
        super().__init__()
 
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.dropout_p = dropout_p

        self.Wq = nn.Linear(d_model, d_model, bias=bias)
        self.Wk = nn.Linear(d_model, d_model, bias=bias)
        self.Wv = nn.Linear(d_model, d_model, bias=bias)
        self.Wo = nn.Linear(d_model, d_model, bias=bias)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        x = x.view(B, T, self.num_heads, self.head_dim)
        return x.transpose(1, 2)

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        B, H, T, D = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(B, T, H * D)

    def forward(self, x_q, x_kv=None, attn_mask=None, need_weights=False):
        if x_kv is None:
            x_kv = x_q

        Q = self._split_heads(self.Wq(x_q))
        K = self._split_heads(self.Wk(x_kv))
        V = self._split_heads(self.Wv(x_kv))

        out, attn = scaled_dot_product_attention(
            Q, K, V,
            attn_mask=attn_mask,
            dropout_p=self.dropout_p,
            training=self.training
        )
        out = self.Wo(self._merge_heads(out))

        if need_weights:
            return out, attn
        return out

In [7]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout_p: float = 0.0):
        super().__init__()
        self.lin1 = nn.Linear(d_model, d_ff)
        self.lin2 = nn.Linear(d_ff, d_model)
        self.dropout_p = dropout_p

    def forward(self, x):
        x = self.lin1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout_p, training=self.training)
        x = self.lin2(x)
        return x

In [8]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = x.size(1)
        return x + self.pe[:T, :].unsqueeze(0)

In [9]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout_p: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout_p=dropout_p)
        self.ffn = FeedForward(d_model, d_ff, dropout_p=dropout_p)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout_p = dropout_p

    def forward(self, x, src_key_padding_mask=None):
        attn_out = self.self_attn(x, attn_mask=src_key_padding_mask)
        x = x + F.dropout(attn_out, p=self.dropout_p, training=self.training)
        x = self.norm1(x)

        ffn_out = self.ffn(x)
        x = x + F.dropout(ffn_out, p=self.dropout_p, training=self.training)
        x = self.norm2(x)
        return x

In [10]:
class Encoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_layers: int, num_heads: int, d_ff: int,
                 dropout_p: float = 0.1, max_len: int = 2048):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = SinusoidalPositionalEncoding(d_model, max_len=max_len)
        self.dropout_p = dropout_p

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout_p=dropout_p)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, src_tokens, src_key_padding_mask=None):
        x = self.tok_emb(src_tokens)
        x = self.pos_emb(x)
        x = F.dropout(x, p=self.dropout_p, training=self.training)

        for layer in self.layers:
            x = layer(x, src_key_padding_mask=src_key_padding_mask)

        return self.norm(x)

In [11]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout_p: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout_p=dropout_p)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout_p=dropout_p)
        self.ffn = FeedForward(d_model, d_ff, dropout_p=dropout_p)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout_p = dropout_p

    def forward(self, x, memory, tgt_attn_mask=None, src_key_padding_mask=None):
        self_out = self.self_attn(x, attn_mask=tgt_attn_mask)
        x = x + F.dropout(self_out, p=self.dropout_p, training=self.training)
        x = self.norm1(x)

        cross_out = self.cross_attn(x_q=x, x_kv=memory, attn_mask=src_key_padding_mask)
        x = x + F.dropout(cross_out, p=self.dropout_p, training=self.training)
        x = self.norm2(x)

        ffn_out = self.ffn(x)
        x = x + F.dropout(ffn_out, p=self.dropout_p, training=self.training)
        x = self.norm3(x)

        return x

In [12]:
class Decoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_layers: int, num_heads: int, d_ff: int,
                 dropout_p: float = 0.1, max_len: int = 2048):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = SinusoidalPositionalEncoding(d_model, max_len=max_len)
        self.dropout_p = dropout_p

        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout_p=dropout_p)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, tgt_tokens, memory, tgt_attn_mask=None, src_key_padding_mask=None):
        x = self.tok_emb(tgt_tokens)
        x = self.pos_emb(x)
        x = F.dropout(x, p=self.dropout_p, training=self.training)

        for layer in self.layers:
            x = layer(x, memory, tgt_attn_mask=tgt_attn_mask, src_key_padding_mask=src_key_padding_mask)

        x = self.norm(x)
        logits = self.lm_head(x)
        return logits

In [13]:
class TransformerSeq2Seq(nn.Module):
    def __init__(self, src_vocab_size: int, tgt_vocab_size: int,
                 d_model: int = 128, num_layers: int = 2,
                 num_heads: int = 4, d_ff: int = 256,
                 dropout_p: float = 0.1, max_len: int = 2048):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, num_layers, num_heads, d_ff, dropout_p, max_len)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_layers, num_heads, d_ff, dropout_p, max_len)

    def forward(self, src_tokens, tgt_tokens, src_pad_mask=None, tgt_causal_mask=None):
        memory = self.encoder(
            src_tokens,
            src_key_padding_mask=src_pad_mask
        )
        
        logits = self.decoder(
            tgt_tokens,
            memory,
            tgt_attn_mask=tgt_causal_mask,
            src_key_padding_mask=src_pad_mask
        )
        return logits

In [14]:
class BPETokenizer:
    def __init__(
        self,
        texts,
        vocab_size=1200,
        pad_token="<pad>",
        bos_token="<bos>",
        eos_token="<eos>",
        unk_token="<unk>",
        space_marker="▁",
        min_pair_freq=2,
    ):
        self.pad_token = pad_token
        self.bos_token = bos_token
        self.eos_token = eos_token
        self.unk_token = unk_token
        self.space_marker = space_marker
        self.min_pair_freq = min_pair_freq
        self.vocab_size = vocab_size

        self.merges = []
        self._train(texts)

    def _normalize(self, text: str) -> str:
        text = re.sub(r"\s+", " ", text.strip())
        if not text:
            return self.space_marker
        return self.space_marker + text.replace(" ", self.space_marker)

    def _train(self, texts):
        corpus = Counter()
        for text in texts:
            normalized = self._normalize(text)
            corpus[tuple(normalized)] += 1

        for _ in tqdm(range(self.vocab_size), desc="Training BPE tokenizer"):
            pair_counts = Counter()
            for tokens, freq in corpus.items():
                for pair in zip(tokens, tokens[1:]):
                    pair_counts[pair] += freq

            if not pair_counts:
                break

            best_pair, best_freq = pair_counts.most_common(1)[0]
            if best_freq < self.min_pair_freq:
                break

            merged_symbol = "".join(best_pair)
            self.merges.append(best_pair)

            new_corpus = Counter()
            for tokens, freq in corpus.items():
                merged_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == best_pair:
                        merged_tokens.append(merged_symbol)
                        i += 2
                    else:
                        merged_tokens.append(tokens[i])
                        i += 1
                new_corpus[tuple(merged_tokens)] += freq
            corpus = new_corpus

            vocab_symbols = set()
            for tokens in corpus:
                vocab_symbols.update(tokens)
            if len(vocab_symbols) + 4 >= self.vocab_size:
                break

        vocab_symbols = set()
        for tokens in corpus:
            vocab_symbols.update(tokens)

        self.itos = [self.pad_token, self.bos_token, self.eos_token, self.unk_token] + sorted(
            vocab_symbols,
            key=lambda x: (len(x), x)
        )
        self.stoi = {token: idx for idx, token in enumerate(self.itos)}

        self.pad_id = self.stoi[self.pad_token]
        self.bos_id = self.stoi[self.bos_token]
        self.eos_id = self.stoi[self.eos_token]
        self.unk_id = self.stoi[self.unk_token]

    def __len__(self):
        return len(self.itos)

    def _apply_merges(self, tokens):
        tokens = list(tokens)
        for pair in self.merges:
            merged_symbol = "".join(pair)
            merged_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                    merged_tokens.append(merged_symbol)
                    i += 2
                else:
                    merged_tokens.append(tokens[i])
                    i += 1
            tokens = merged_tokens
        return tokens

    def encode(self, text: str, add_special=True):
        normalized = self._normalize(text)
        pieces = self._apply_merges(list(normalized))
        ids = [self.stoi.get(piece, self.unk_id) for piece in pieces]
        if add_special:
            ids = [self.bos_id] + ids + [self.eos_id]
        return ids

    def decode(self, ids):
        pieces = []
        for idx in ids:
            if idx >= len(self.itos):
                continue
            piece = self.itos[idx]
            if piece in (self.pad_token, self.bos_token, self.eos_token):
                continue
            if piece == self.unk_token:
                pieces.append("")
            else:
                pieces.append(piece)

        text = "".join(pieces).replace(self.space_marker, " ").strip()
        return re.sub(r"\s+", " ", text)


In [15]:
def pad_to_max(batch_ids, pad_id):
    max_len = max(len(x) for x in batch_ids)
    out = []
    for x in batch_ids:
        out.append(x + [pad_id] * (max_len - len(x)))
    return torch.tensor(out, dtype=torch.long)

@torch.no_grad()
def next_token_distribution(model, src_tokenizer, tgt_tokenizer, src_text: str, tgt_prefix: str):
    model.eval()

    src = torch.tensor([src_tokenizer.encode(src_text)], device=device)
    tgt = torch.tensor([tgt_tokenizer.encode(tgt_prefix)], device=device)

    src_pad = make_padding_mask(src, src_tokenizer.pad_id)
    Tt = tgt.size(1)
    causal = make_causal_mask(Tt, device=device)

    logits = model(src, tgt, src_pad_mask=src_pad, tgt_causal_mask=causal)
    last_logits = logits[:, -1, :]
    probs = torch.softmax(last_logits, dim=-1)[0]
    return probs

def train_step(
    model,
    optimizer,
    loss_fn,
    scaler,
    src,
    tgt,
    src_pad_id,
    tgt_pad_id,
    scheduler=None,
    grad_clip=1.0,
):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    tgt_in = tgt[:, :-1]
    tgt_out = tgt[:, 1:]

    src_pad_mask = make_padding_mask(src, src_pad_id)
    tgt_pad_mask = make_padding_mask(tgt_in, tgt_pad_id)
    tgt_causal_mask = make_causal_mask(tgt_in.size(1), device=src.device)
    tgt_mask = tgt_pad_mask * tgt_causal_mask

    with autocast(enabled=torch.cuda.is_available()):
        logits = model(
            src,
            tgt_in,
            src_pad_mask=src_pad_mask,
            tgt_causal_mask=tgt_mask
        )
        B, T, V = logits.shape
        loss = loss_fn(logits.reshape(B * T, V), tgt_out.reshape(B * T))

    scaler.scale(loss).backward()

    if grad_clip is not None:
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        grad_norm = float(grad_norm)
    else:
        grad_norm = None

    scaler.step(optimizer)
    scaler.update()

    if scheduler is not None:
        scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    return loss.item(), grad_norm, current_lr


In [ ]:
@torch.no_grad()
def evaluate_validation_loss(model, valid_loader, src_pad_id, tgt_pad_id):
    model.eval()
    total_loss = 0.0
    total_batches = 0
    total_correct = 0
    total_tokens = 0

    loss_fn = nn.CrossEntropyLoss(ignore_index=tgt_pad_id)

    for src, tgt in valid_loader:
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        src_pad_mask = make_padding_mask(src, src_pad_id)
        tgt_pad_mask = make_padding_mask(tgt_in, tgt_pad_id)
        tgt_causal_mask = make_causal_mask(tgt_in.size(1), device=src.device)
        tgt_mask = tgt_pad_mask * tgt_causal_mask

        logits = model(
            src,
            tgt_in,
            src_pad_mask=src_pad_mask,
            tgt_causal_mask=tgt_mask
        )

        B, T, V = logits.shape
        loss = loss_fn(logits.reshape(B * T, V), tgt_out.reshape(B * T))
        total_loss += loss.item()
        total_batches += 1

        preds = logits.argmax(dim=-1)
        non_pad = tgt_out != tgt_pad_id
        total_correct += ((preds == tgt_out) & non_pad).sum().item()
        total_tokens += non_pad.sum().item()

    avg_loss = total_loss / max(total_batches, 1)
    token_accuracy = total_correct / max(total_tokens, 1)
    perplexity = float(np.exp(avg_loss))
    return {
        "val_loss": avg_loss,
        "val_token_accuracy": token_accuracy,
        "val_perplexity": perplexity,
    }

def run_train(
    model,
    train_loader,
    valid_loader,
    epochs,
    src_tok,
    tgt_tok,
    checkpoint_path="transformer_checkpoint.pt",
    lr=3e-4,
    weight_decay=0.01,
    label_smoothing=0.1,
    warmup_ratio=0.1,
    min_lr_ratio=0.1,
    grad_clip=1.0,
    patience=5,
):
    scaler = GradScaler(enabled=torch.cuda.is_available())

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        betas=(0.9, 0.98),
        weight_decay=weight_decay,
    )
    loss_fn = nn.CrossEntropyLoss(
        ignore_index=tgt_tok.pad_id,
        label_smoothing=label_smoothing,
    )

    total_steps = epochs * len(train_loader)
    warmup_steps = max(1, int(total_steps * warmup_ratio))
    cosine_steps = max(1, total_steps - warmup_steps)

    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[
            torch.optim.lr_scheduler.LinearLR(
                optimizer,
                start_factor=min_lr_ratio,
                end_factor=1.0,
                total_iters=warmup_steps,
            ),
            torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=cosine_steps,
                eta_min=lr * min_lr_ratio,
            ),
        ],
        milestones=[warmup_steps],
    )

    best_val_loss = float("inf")
    best_epoch = 0
    start_epoch = 1
    history = []

    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)

        same_src_vocab = checkpoint.get("src_vocab_size") == model.encoder.tok_emb.num_embeddings
        same_tgt_vocab = checkpoint.get("tgt_vocab_size") == model.decoder.tok_emb.num_embeddings

        if same_src_vocab and same_tgt_vocab:
            model.load_state_dict(checkpoint["model_state_dict"])
            if "optimizer_state_dict" in checkpoint:
                optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            if "scheduler_state_dict" in checkpoint:
                scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
            if "scaler_state_dict" in checkpoint and torch.cuda.is_available():
                scaler.load_state_dict(checkpoint["scaler_state_dict"])

            best_val_loss = checkpoint.get("best_val_loss", best_val_loss)
            best_epoch = checkpoint.get("best_epoch", best_epoch)
            history = checkpoint.get("history", history)
            start_epoch = checkpoint.get("epoch", 0) + 1

            print(f"Resume from epoch {start_epoch}. Best val_loss={best_val_loss:.4f}")
        else:
            print("Checkpoint пропущен: размер словаря не совпадает.")

    no_improve_epochs = 0
    progress_bar = tqdm(range(start_epoch, epochs + 1), desc="Training")

    for epoch in progress_bar:
        model.train()
        epoch_loss = 0.0
        epoch_steps = 0
        last_grad_norm = None
        last_lr = optimizer.param_groups[0]["lr"]

        for src, tgt in train_loader:
            src = src.to(device)
            tgt = tgt.to(device)

            batch_loss, grad_norm, current_lr = train_step(
                model=model,
                optimizer=optimizer,
                loss_fn=loss_fn,
                scaler=scaler,
                src=src,
                tgt=tgt,
                src_pad_id=src_tok.pad_id,
                tgt_pad_id=tgt_tok.pad_id,
                scheduler=scheduler,
                grad_clip=grad_clip,
            )

            epoch_loss += batch_loss
            epoch_steps += 1
            last_grad_norm = grad_norm
            last_lr = current_lr

        train_loss = epoch_loss / max(epoch_steps, 1)
        val_metrics = evaluate_validation_loss(model, valid_loader, src_tok.pad_id, tgt_tok.pad_id)
        val_loss = val_metrics["val_loss"]

        epoch_record = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "val_perplexity": float(val_metrics["val_perplexity"]),
            "val_token_accuracy": float(val_metrics["val_token_accuracy"]),
            "lr": float(last_lr),
            "grad_norm": None if last_grad_norm is None else float(last_grad_norm),
        }
        history.append(epoch_record)

        improved = val_loss < best_val_loss
        if improved:
            best_val_loss = val_loss
            best_epoch = epoch
            no_improve_epochs = 0

            torch.save({
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_val_loss": best_val_loss,
                "history": history,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "src_vocab_size": model.encoder.tok_emb.num_embeddings,
                "tgt_vocab_size": model.decoder.tok_emb.num_embeddings,
                "config": {
                    "lr": lr,
                    "weight_decay": weight_decay,
                    "label_smoothing": label_smoothing,
                    "warmup_ratio": warmup_ratio,
                    "min_lr_ratio": min_lr_ratio,
                    "grad_clip": grad_clip,
                    "patience": patience,
                },
            }, checkpoint_path)
        else:
            no_improve_epochs += 1

        progress_bar.set_postfix({
            "train_loss": f"{train_loss:.4f}",
            "val_loss": f"{val_loss:.4f}",
            "ppl": f"{val_metrics['val_perplexity']:.2f}",
            "acc": f"{val_metrics['val_token_accuracy']:.3f}",
            "lr": f"{last_lr:.2e}",
            "best": best_epoch,
        })

        print(
            f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | val_ppl={val_metrics['val_perplexity']:.2f} | "
            f"val_acc={val_metrics['val_token_accuracy']:.4f} | lr={last_lr:.2e} | "
            f"{'saved best' if improved else f'no improve {no_improve_epochs}/{patience}'}"
        )

        if no_improve_epochs >= patience:
            print(f"Early stopping: на протяжении {patience} эпох не было улучшения val_loss.")
            break

    if os.path.exists(checkpoint_path):
        best_checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(best_checkpoint["model_state_dict"])
        model.training_history = best_checkpoint.get("history", history)
        model.best_val_loss = best_checkpoint.get("best_val_loss", best_val_loss)
        model.best_epoch = best_checkpoint.get("best_epoch", best_epoch)
    else:
        model.training_history = history
        model.best_val_loss = best_val_loss
        model.best_epoch = best_epoch

    return model


In [17]:
dataset = load_dataset("opus_books", "en-ru")

full_train = dataset["train"]

split = full_train.train_test_split(test_size=0.1, seed=42)

train_raw = split["train"]
valid_raw = split["test"]

train_pairs = [
    (x["translation"]["en"].strip(), x["translation"]["ru"].strip())
    for x in train_raw
]

valid_pairs = [
    (x["translation"]["en"].strip(), x["translation"]["ru"].strip())
    for x in valid_raw
]

In [18]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_tok, tgt_tok, max_src_len=96, max_tgt_len=96):
        self.src_tok = src_tok
        self.tgt_tok = tgt_tok
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

        self.encoded_pairs = []
        for src_text, tgt_text in tqdm(pairs, desc="Encoding dataset"):
            src_ids = self.src_tok.encode(src_text)[:self.max_src_len]
            tgt_ids = self.tgt_tok.encode(tgt_text)[:self.max_tgt_len]
            if len(src_ids) >= 2 and len(tgt_ids) >= 2:
                self.encoded_pairs.append((src_ids, tgt_ids, src_text, tgt_text))

    def __len__(self):
        return len(self.encoded_pairs)

    def __getitem__(self, idx):
        src_ids, tgt_ids, _, _ = self.encoded_pairs[idx]
        return src_ids, tgt_ids

def make_collate_fn(src_pad_id, tgt_pad_id):
    def collate_fn(batch):
        src_batch, tgt_batch = zip(*batch)
        src = pad_to_max(src_batch, src_pad_id)
        tgt = pad_to_max(tgt_batch, tgt_pad_id)
        return src, tgt
    return collate_fn


In [19]:
tokenizer_train_pairs = train_pairs[:30000]
train_src_texts = [src for src, _ in tokenizer_train_pairs]
train_tgt_texts = [tgt for _, tgt in tokenizer_train_pairs]

src_tok = BPETokenizer(train_src_texts, vocab_size=1000)
tgt_tok = BPETokenizer(train_tgt_texts, vocab_size=1200)

print(f"Source vocab size: {len(src_tok)}")
print(f"Target vocab size: {len(tgt_tok)}")
print("Source encode example:", src_tok.encode(train_src_texts[0])[:20])
print("Source decode example:", src_tok.decode(src_tok.encode(train_src_texts[0])))

train_ds = TranslationDataset(train_pairs[:30000], src_tok, tgt_tok, max_src_len=96, max_tgt_len=96)
valid_ds = TranslationDataset(valid_pairs[:2000], src_tok, tgt_tok, max_src_len=96, max_tgt_len=96)

collate_fn = make_collate_fn(src_tok.pad_id, tgt_tok.pad_id)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)


Training BPE tokenizer:  89%|████████▊ | 1064/1200 [07:40<00:58,  2.31it/s]


Source vocab size: 1000
Target vocab size: 1200
Source encode example: [1, 832, 977, 827, 75, 217, 577, 939, 243, 567, 683, 576, 737, 152, 2]
Source decode example: She thought with wonder of her state the day before.


Encoding dataset: 100%|██████████| 1750/1750 [00:30<00:00, 57.29it/s]


In [20]:
valid_loader = DataLoader(valid_ds, batch_size=32, shuffle=False, collate_fn=collate_fn)
len(train_loader), len(valid_loader)

(493, 55)

In [21]:
model = TransformerSeq2Seq(
    src_vocab_size=len(src_tok),
    tgt_vocab_size=len(tgt_tok),
    d_model=256,
    num_layers=4,
    num_heads=8,
    d_ff=1024,
    dropout_p=0.1,
).to(device)


In [ ]:
model = run_train(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    epochs=30,
    src_tok=src_tok,
    tgt_tok=tgt_tok,
    checkpoint_path="transformer_checkpoint.pt",
    lr=3e-4,
    weight_decay=0.01,
    label_smoothing=0.1,
    warmup_ratio=0.1,
    min_lr_ratio=0.1,
    grad_clip=1.0,
    patience=5,
)

if hasattr(model, "training_history") and len(model.training_history) > 0:
    print(f"Best epoch: {model.best_epoch}, best val_loss: {model.best_val_loss:.4f}")
    print("Last history record:", model.training_history[-1])


C:\Users\semen\AppData\Local\Temp\ipykernel_17696\3169870037.py:65: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())
Training:   0%|          | 0/30 [00:00<?, ?it/s]C:\Users\semen\AppData\Local\Temp\ipykernel_17696\3910281857.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):
Training:   3%|▎         | 1/30 [01:48<52:34, 108.79s/it, train_loss=6.4505, val_loss=5.8016, ppl=330.83, acc=0.072, lr=1.20e-04, best=1]

Epoch 01 | train_loss=6.4505 | val_loss=5.8016 | val_ppl=330.83 | val_acc=0.0718 | lr=1.20e-04 | saved best


Training:   7%|▋         | 2/30 [03:36<50:32, 108.30s/it, train_loss=5.5847, val_loss=4.7977, ppl=121.23, acc=0.140, lr=2.10e-04, best=2]

Epoch 02 | train_loss=5.5847 | val_loss=4.7977 | val_ppl=121.23 | val_acc=0.1399 | lr=2.10e-04 | saved best


c:\Users\semen\works\NN2\venv\Lib\site-packages\torch\optim\lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)
Training:  10%|█         | 3/30 [05:25<48:44, 108.33s/it, train_loss=5.0144, val_loss=4.3773, ppl=79.62, acc=0.167, lr=3.00e-04, best=3] 

...

Epoch 27 | train_loss=2.9774 | val_loss=3.1591 | val_ppl=23.55 | val_acc=0.3600 | lr=3.81e-05 | no improve 3/5


Training:  93%|█████████▎| 28/30 [51:18<03:40, 110.44s/it, train_loss=2.9608, val_loss=3.1572, ppl=23.50, acc=0.360, lr=3.36e-05, best=24]

Epoch 28 | train_loss=2.9608 | val_loss=3.1572 | val_ppl=23.50 | val_acc=0.3600 | lr=3.36e-05 | no improve 4/5


Training:  93%|█████████▎| 28/30 [53:09<03:47, 113.90s/it, train_loss=2.9509, val_loss=3.1597, ppl=23.56, acc=0.361, lr=3.09e-05, best=24]
C:\Users\semen\AppData\Local\Temp\ipykernel_17696\3169870037.py:227: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Pleas

Epoch 29 | train_loss=2.9509 | val_loss=3.1597 | val_ppl=23.56 | val_acc=0.3608 | lr=3.09e-05 | no improve 5/5
Early stopping: на протяжении 5 эпох не было улучшения val_loss.
Best epoch: 24, best val_loss: 3.1537
Last history record: {'epoch': 24, 'train_loss': 3.036541570272697, 'val_loss': 3.153650175441395, 'val_perplexity': 23.42140096240657, 'val_token_accuracy': 0.358301487592462, 'lr': 6.15840001789464e-05, 'grad_norm': 6.977954387664795}



## Оценка качества модели


In [ ]:

from collections import Counter
from math import exp


def _extract_ngrams(tokens, n):
    if len(tokens) < n:
        return Counter()
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))


def corpus_bleu_score(references, hypotheses, max_n=4, smooth=1.0):
    clipped_counts = [0] * max_n
    total_counts = [0] * max_n
    ref_len = 0
    hyp_len = 0

    for ref, hyp in zip(references, hypotheses):
        ref_tokens = ref.split()
        hyp_tokens = hyp.split()
        ref_len += len(ref_tokens)
        hyp_len += len(hyp_tokens)

        for n in range(1, max_n + 1):
            ref_ngrams = _extract_ngrams(ref_tokens, n)
            hyp_ngrams = _extract_ngrams(hyp_tokens, n)
            total_counts[n - 1] += max(sum(hyp_ngrams.values()), 0)
            for ng, count in hyp_ngrams.items():
                clipped_counts[n - 1] += min(count, ref_ngrams.get(ng, 0))

    precisions = []
    for clip, total in zip(clipped_counts, total_counts):
        precisions.append((clip + smooth) / (total + smooth))

    if hyp_len == 0:
        return 0.0
    bp = 1.0 if hyp_len > ref_len else np.exp(1 - ref_len / max(hyp_len, 1))
    bleu = bp * np.exp(sum(np.log(p) for p in precisions) / max_n)
    return bleu * 100


def corpus_chrf_score(references, hypotheses, max_n=6, beta=2.0, eps=1e-12):
    total_prec = []
    total_rec = []

    for n in range(1, max_n + 1):
        overlap = 0
        hyp_total = 0
        ref_total = 0
        for ref, hyp in zip(references, hypotheses):
            ref_chars = list(ref)
            hyp_chars = list(hyp)
            ref_ngrams = _extract_ngrams(ref_chars, n)
            hyp_ngrams = _extract_ngrams(hyp_chars, n)
            overlap += sum(min(count, ref_ngrams.get(ng, 0)) for ng, count in hyp_ngrams.items())
            hyp_total += sum(hyp_ngrams.values())
            ref_total += sum(ref_ngrams.values())

        prec = overlap / (hyp_total + eps)
        rec = overlap / (ref_total + eps)
        total_prec.append(prec)
        total_rec.append(rec)

    mean_prec = sum(total_prec) / max_n
    mean_rec = sum(total_rec) / max_n
    beta2 = beta ** 2
    chrf = (1 + beta2) * mean_prec * mean_rec / (beta2 * mean_prec + mean_rec + eps)
    return chrf * 100


def lcs_length(a, b):
    dp = [0] * (len(b) + 1)
    for i in range(1, len(a) + 1):
        prev = 0
        for j in range(1, len(b) + 1):
            cur = dp[j]
            if a[i - 1] == b[j - 1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = cur
    return dp[-1]


def corpus_rouge_l(references, hypotheses, eps=1e-12):
    scores = []
    for ref, hyp in zip(references, hypotheses):
        ref_tokens = ref.split()
        hyp_tokens = hyp.split()
        if not ref_tokens or not hyp_tokens:
            scores.append(0.0)
            continue
        lcs = lcs_length(ref_tokens, hyp_tokens)
        prec = lcs / (len(hyp_tokens) + eps)
        rec = lcs / (len(ref_tokens) + eps)
        f1 = 2 * prec * rec / (prec + rec + eps)
        scores.append(f1)
    return 100 * sum(scores) / max(len(scores), 1)


def levenshtein_distance(a, b):
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        prev = dp[0]
        dp[0] = i
        for j, cb in enumerate(b, start=1):
            cur = dp[j]
            if ca == cb:
                dp[j] = prev
            else:
                dp[j] = 1 + min(prev, dp[j], dp[j - 1])
            prev = cur
    return dp[-1]


def normalized_edit_similarity(references, hypotheses):
    sims = []
    for ref, hyp in zip(references, hypotheses):
        denom = max(len(ref), len(hyp), 1)
        sims.append(1.0 - levenshtein_distance(ref, hyp) / denom)
    return 100 * sum(sims) / max(len(sims), 1)


@torch.no_grad()
def greedy_decode(model, src_text, src_tok, tgt_tok, max_new_tokens=160):
    model.eval()
    src = torch.tensor([src_tok.encode(src_text)], dtype=torch.long, device=device)
    src_pad_mask = make_padding_mask(src, src_tok.pad_id)

    generated = [tgt_tok.bos_id]
    for _ in range(max_new_tokens):
        tgt = torch.tensor([generated], dtype=torch.long, device=device)
        tgt_mask = make_causal_mask(tgt.size(1), device=device)
        logits = model(src, tgt, src_pad_mask=src_pad_mask, tgt_causal_mask=tgt_mask)
        next_id = logits[:, -1, :].argmax(dim=-1).item()
        generated.append(next_id)
        if next_id == tgt_tok.eos_id:
            break

    return tgt_tok.decode(generated)


@torch.no_grad()
def evaluate_teacher_forcing(model, data_loader, src_tok, tgt_tok):
    model.eval()
    loss_fn = nn.CrossEntropyLoss(ignore_index=tgt_tok.pad_id, reduction='sum')

    total_loss = 0.0
    total_tokens = 0
    correct_tokens = 0

    for src, tgt in tqdm(data_loader, desc='Teacher-forcing evaluation'):
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        src_pad_mask = make_padding_mask(src, src_tok.pad_id)
        tgt_pad_mask = make_padding_mask(tgt_in, tgt_tok.pad_id)
        tgt_causal_mask = make_causal_mask(tgt_in.size(1), device=src.device)
        tgt_mask = tgt_pad_mask * tgt_causal_mask

        logits = model(src, tgt_in, src_pad_mask=src_pad_mask, tgt_causal_mask=tgt_mask)
        B, T, V = logits.shape

        total_loss += loss_fn(logits.reshape(B * T, V), tgt_out.reshape(B * T)).item()

        preds = logits.argmax(dim=-1)
        valid_mask = (tgt_out != tgt_tok.pad_id)
        correct_tokens += ((preds == tgt_out) & valid_mask).sum().item()
        total_tokens += valid_mask.sum().item()

    avg_loss = total_loss / max(total_tokens, 1)
    perplexity = float(np.exp(avg_loss))
    token_acc = correct_tokens / max(total_tokens, 1)

    return {
        'val_loss': avg_loss,
        'perplexity': perplexity,
        'token_accuracy': token_acc,
    }


@torch.no_grad()
def evaluate_generation_metrics(model, pairs, src_tok, tgt_tok, max_samples=200, print_examples=5):
    model.eval()
    references = []
    hypotheses = []

    sampled_pairs = pairs[:max_samples]
    for src_text, tgt_text in tqdm(sampled_pairs, desc='Generation evaluation'):
        pred = greedy_decode(model, src_text, src_tok, tgt_tok)
        references.append(tgt_text)
        hypotheses.append(pred)

    bleu = corpus_bleu_score(references, hypotheses)
    chrf = corpus_chrf_score(references, hypotheses)
    rouge_l = corpus_rouge_l(references, hypotheses)
    exact_match = 100 * np.mean([ref == hyp for ref, hyp in zip(references, hypotheses)])
    edit_sim = normalized_edit_similarity(references, hypotheses)

    metrics = {
        'BLEU-4': bleu,
        'chrF': chrf,
        'ROUGE-L': rouge_l,
        'Exact Match': exact_match,
        'Normalized Edit Similarity': edit_sim,
    }

    print('Примеры предсказаний:')
    for i, (src_text, ref, hyp) in enumerate(zip([x[0] for x in sampled_pairs], references, hypotheses)):
        if i >= print_examples:
            break
        print(f'\n[{i+1}] SRC: {src_text}')
        print(f'REF: {ref}')
        print(f'HYP: {hyp}')

    return metrics


In [25]:
teacher_metrics = evaluate_teacher_forcing(model, valid_loader, src_tok, tgt_tok)
print('Teacher-forcing metrics:')
for k, v in teacher_metrics.items():
    if isinstance(v, float):
        print(f'{k}: {v:.4f}')
    else:
        print(f'{k}: {v}')

generation_metrics = evaluate_generation_metrics(
    model,
    valid_pairs,
    src_tok,
    tgt_tok,
    max_samples=200,
    print_examples=5,
)

print('\nGeneration metrics:')
for k, v in generation_metrics.items():
    print(f'{k}: {v:.4f}')


Teacher-forcing evaluation: 100%|██████████| 55/55 [00:04<00:00, 13.51it/s]


Teacher-forcing metrics:
val_loss: 3.1606
perplexity: 23.5850
token_accuracy: 0.3583


Generation evaluation: 100%|██████████| 200/200 [01:06<00:00,  3.00it/s]


Примеры предсказаний:

[1] SRC: 'That is my position. You may trample me in the mud, make me the laughing-stock of the world, – I will not forsake her and will never utter a word of reproach to you,' continued Karenin.
REF: Вы можете затоптать меня в грязь, сделать посмешищем света, я не покину ее и никогда слова упрека не скажу вам, -- продолжал он. -- Моя обязанность ясно начертана для меня: я должен быть с ней и буду.
HYP: -- Это мое положение, ты заставишь меня, -- смеясь, сказал Алексей Александрович. -- Ты меня никам хужебе не заставить ее слово, -- никогда не продолжал Алексей Александрович. -- Он меня никогда не продолжал много положения.

[2] SRC: While they were talking Laska, pricking her ears, kept looking up at the sky and then reproachfully at them.
REF: В то время, как они говорили это, Ласка, насторожив уши, оглядывалась вверх на небо и укоризненно на них.
HYP: Пока они говорили о Ласках ках какатка, поспешно смотрела на него, и в смотря на то.

[3] SRC: His nearness wa